# 具身强化学习实验室：从 Bellman 到 PPO、SAC 与世界模型

同一个机器人先取货、再送达。逐格执行，每次只替换一个学习模块。
完整解释见 `COURSE.md`；一键批量运行见 `run_suite.py`。

这里使用与命令行一致的真实代码，不另写一套简化算法。神经训练单元可能需要几十秒至数分钟。

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "warehouse").exists():
    ROOT = ROOT / "具身强化学习实验室"
assert (ROOT / "warehouse").exists(), "请在具身强化学习实验室项目目录或桌面启动 notebook"
sys.path.insert(0, str(ROOT))
if (ROOT / ".deps").exists():
    sys.path.insert(0, str(ROOT / ".deps"))
import numpy as np
import torch
import matplotlib.pyplot as plt
from warehouse.env import Warehouse, MAP, ACTION_NAMES, evaluate
from warehouse.tabular import value_iteration, policy_evaluation, prediction, control
print("Python:", sys.version.split()[0], "PyTorch:", torch.__version__)
from warehouse.device import resolve_device
DEVICE = "auto"  # change to "cpu" or "cuda"
COMPUTE_DEVICE = resolve_device(DEVICE)
print("Training device:", COMPUTE_DEVICE)

Python: 3.12.11 PyTorch: 2.8.0+cu126
Training device: cuda


## 先认识状态与观测
同样的位置，取货前后动作目标不同。因此 `carrying` 必须进入状态。蓝色机器人未载货，紫色机器人载货。RGB 是未来世界模型可以使用的另一种观测形式。

In [ ]:
env = Warehouse(seed=7)
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for phase, ax in enumerate(axes):
    s = env.encode(2, 0, phase)
    ax.imshow(env.observation(s, "rgb"))
    ax.set_title(f"carrying={phase}")
    ax.axis("off")
plt.show()
print("state count:", env.n_states, "valid nonterminal:", len(env.valid_states))

## 一条真实轨迹的奖励与回报
为了先能手算，这个单元临时关闭打滑，其余课程默认 10% 打滑。
路径为下 4 次、右 6 次、上 4 次。对比即时奖励、原始累计回报与塑形回报。到达 P 只奖励一次。

In [ ]:
hand_env = Warehouse(seed=7, slip=0)
s = hand_env.reset()
trajectory = []
for a in [2]*4 + [1]*6 + [0]*4:
    tr = hand_env.step(a)
    trajectory.append(tr)
    print(hand_env.decode(tr.state), ACTION_NAMES[a], "->", hand_env.decode(tr.next_state),
          "base=", round(tr.base_reward, 3), "train=", round(tr.reward, 3))
    if tr.terminated:
        break
base_return = sum(hand_env.gamma**t * tr.base_reward for t,tr in enumerate(trajectory))
train_return = sum(hand_env.gamma**t * tr.reward for t,tr in enumerate(trajectory))
print("base return:", base_return, "training return:", train_return)
print("difference:", train_return-base_return, "= -Phi(start):", -hand_env.potential(hand_env.reset()))

## 1. Bellman：先用已知模型建立独立参考
`outcomes` 是环境模型，只有规划与参考检查可以直接调用。后面 Q-learning 和 DQN 只能使用交互样本。

In [ ]:
env = Warehouse(seed=7)
v_star, q_star = value_iteration(env)
start = env.reset()
print("V*(start), shaped:", v_star[start])
print("Q*(start):", dict(zip(ACTION_NAMES, q_star[start])))
print("independent evaluation:", evaluate(lambda s: int(q_star[s].argmax()), episodes=200))

## 2–3. Monte Carlo 与 TD 评价同一个固定策略
这一步不改进策略。比较 `start_estimate` 与精确 `start_exact`。不要把固定策略的成功率理解成 MC/TD 的策略训练效果。

In [ ]:
prediction_logs = {}
for method in ["mc", "td"]:
    value, logs, fixed_policy = prediction(method, episodes=3000, seed=7)
    prediction_logs[method] = logs
    print(method, logs[-1])
for method, logs in prediction_logs.items():
    plt.plot([x["steps"] for x in logs],
             [abs(x["start_estimate"]-x["start_exact"]) for x in logs], label=method)
plt.xlabel("Environment steps"); plt.ylabel("Start-value absolute error"); plt.legend(); plt.show()

## 4–5. SARSA 与 Q-learning：只改下一动作评价
SARSA 用下一次行为策略实际采样的动作；Q-learning 用下一状态最大的 Q。
两者训练时都有探索，最终独立测试使用贪心动作。

In [ ]:
tables = {}
for method in ["sarsa", "q"]:
    q, logs = control(method, episodes=3000, seed=7)
    tables[method] = q
    print(method, evaluate(lambda s: int(q[s].argmax()), episodes=200))
tr = trajectory[-2]
q = tables["q"]
print("An actual transition:", tr)
print("Q-learning target:", tr.reward + env.gamma*(not tr.terminated)*q[tr.next_state].max())

## 6. DQN：把表换成网络
运行真实入口，保存网络、回放轨迹与曲线。`--steps` 是环境交互预算，512 步 warmup 后每 4 步更新一次。

In [ ]:
import subprocess
subprocess.run([sys.executable, str(ROOT/"run.py"), "dqn", "--steps", "65536", "--device", DEVICE], check=True)

## 7. REINFORCE：完整回报直接更新 Actor
每 8 个完整回合更新一次，无神经 Critic。注意它的 `--episodes` 与 DQN 的 `--steps` 是不同预算单位。

In [ ]:
subprocess.run([sys.executable, str(ROOT/"run.py"), "reinforce", "--episodes", "2000", "--device", DEVICE], check=True)

## 8–10. AC → GAE → PPO
共用网络、16 个环境与采样框架：AC 用一步 delta，GAE 换成多步优势，PPO 再增加概率比率、裁剪和多轮更新。
先手算 GAE，再运行三种方法。

In [ ]:
from warehouse.neural import categorical_gae
r = torch.tensor([[0.],[0.],[10.]])
v = torch.tensor([[6.],[7.],[8.]])
nv = torch.tensor([[7.],[8.],[0.]])
done = torch.tensor([[0.],[0.],[1.]])
A = categorical_gae(r, v, nv, done, done, .9, .95)
print("GAE:", A[:,0].tolist(), "expected: [1.93305, 1.91, 2]")
print("TD:", categorical_gae(r, v, nv, done, done, .9, 0)[:,0].tolist())

In [ ]:
for method in ["ac", "gae", "ppo"]:
    subprocess.run([sys.executable, str(ROOT/"run.py"), method, "--steps", "65536", "--device", DEVICE], check=True)

## 11. 离散 SAC：双 Q + 随机 Actor + 熵温度
同一离散任务中，对四个动作精确求和，不使用连续版的 tanh-Gaussian。观察 learning.csv 中的 temperature。


In [ ]:
subprocess.run([sys.executable, str(ROOT/"run.py"), "sac", "--steps", "65536", "--device", DEVICE], check=True)

## 汇总结果，但不要用单随机种子排算法名次
控制成功率与固定策略预测误差分开画图。还应比较真实交互预算、原始回报、危险率和不同种子的波动。

In [ ]:
subprocess.run([sys.executable, str(ROOT/"summarize.py")], check=True)
print((ROOT/"output"/"RESULTS.md").read_text(encoding="utf-8"))
plt.figure(figsize=(14,5))
plt.imshow(plt.imread(ROOT/"output"/"learning_curves.png")); plt.axis("off"); plt.show()

## 世界模型桥梁：学下一状态分布、奖励与 continuation
这是 fully observed 状态模型，不是 Dreamer。先训练与验证预测，再检查不跨回合的序列数据。

In [ ]:
subprocess.run([sys.executable, str(ROOT/"run.py"), "world-model", "--episodes", "300", "--steps", "2000", "--device", DEVICE], check=True)
import json
print(json.loads((ROOT/"output"/"world-model_seed7"/"report.json").read_text()))
from warehouse.replay import EpisodeStore
store = EpisodeStore.load(ROOT/"output"/"world-model_seed7"/"episodes.json")
seq = store.sample_sequences(batch_size=2, length=16)
print({key: value.shape for key,value in seq.items()})
print("valid lengths:", seq["mask"].sum(axis=1))

## 交互式查看关键过程
生成离线页面并在浏览器中打开，可逐步回放、查看 Bellman 更新、训练曲线和世界模型预测。诊断数值在冻结模型上重算，界面会明确区分教学演示与实际训练日志。


In [ ]:
subprocess.run([sys.executable, str(ROOT/"visualize.py"), "--open"], check=True)

## 后续实验
1. 关闭奖励塑形，看哪些方法需要更多探索。
2. 把打滑概率从 0.1 改为 0.25，分析危险区附近的策略变化；所有方法要共同修改同一环境定义。
3. 更换随机种子，统一交互预算再比较。
4. 从状态模型扩展到短期模型规划，再换成 RGB 编码与潜在序列模型。具体边界和 Dreamer 路线见 `WORLD_MODELS.md`。


## 交互式教科书：理论与实践一起学

打开 `output/dashboard.html`，选择「理论 → 代码」。17 课按 Bellman、MC、TD、SARSA/Q、DQN、Actor-Critic、GAE、PPO、连续 SAC、TD3、HER、BC、IQL、MPC、动力学随机化、记忆策略、框架衔接组织。

每课包含：实际问题、推导步骤、公式符号与代码变量对照、从当前源文件提取的代码及行号、可调数值实验、可复现命令、观察建议和自测。浏览器数值演算不等于重新训练；已保存策略的回放、教学演算和框架短跑分别注明。

连续控制实战在 `embodied/`，是带惯性和动作延迟的二维对接子技能。参见 [具身技术覆盖与实践](EMBODIED_TECHNIQUES.md)。

```powershell
python visualize.py
python run_embodied.py sac --steps 20000 --device auto
python run_embodied.py her --steps 20000 --device auto
python run_embodied.py sac --sparse --steps 20000 --device auto
python framework_doctor.py
python run_framework.py sac --env Reacher-v5 --steps 512 --device auto
```

先预测参数改变的结果，再操作滑块，最后运行训练命令核对。对照实验应固定奖励、预算、环境和种子；多种子才能支持稳定的算法比较。保存的单种子实验不能证明算法普遍优劣。稀疏 SAC 的失败、BC 数据覆盖不足的失败也保留供分析。

完整 Dreamer/RSSM、真实机器人部署、视觉策略、Isaac Lab/ManiSkill、ACT/Diffusion/VLA 等仍标为后续扩展，未冒充已经实现。当前世界模型包含离散一步预测和连续动力学模型加 MPC。


In [ ]:
# 从公式到当前源文件：不启动训练
from pathlib import Path
from textbook import build_lessons
lessons = build_lessons(Path.cwd())
[(x["title"], x["code"]["path"], x["code"]["line"]) for x in lessons]
